In [1]:
import os
import numpy as np
import librosa
import tensorflow as tf
import joblib

# PATHS
MODEL_PATH = "../models/emotion_model.h5"
SCALER_PATH = "../models/scaler.pkl"
CLASSES_PATH = "../data/processed/classes.npy"

# Load Artifacts
model = tf.keras.models.load_model(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)
classes = np.load(CLASSES_PATH)

print("✅ Brain loaded successfully.")
print(f"Classes: {classes}")

✅ Brain loaded successfully.
Classes: ['angry' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']


In [2]:
# Preprocessing Function
# CONFIG MUST MATCH TRAINING
SAMPLE_RATE = 22050
DURATION = 4 
SAMPLES_PER_TRACK = SAMPLE_RATE * DURATION

def process_audio(file_path):
    # 1. Load
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE)

    # 2. Pad/Truncate
    if len(y) > SAMPLES_PER_TRACK:
        y = y[:SAMPLES_PER_TRACK]
    else:
        padding = SAMPLES_PER_TRACK - len(y)
        y = np.pad(y, (0, padding), mode='constant')

    # 3. Extract Features (COPY FROM NOTEBOOK 02)
    mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
    stft = np.abs(librosa.stft(y))
    chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sr).T, axis=0)
    mel = np.mean(librosa.feature.melspectrogram(y=y, sr=sr).T, axis=0)
    contrast = np.mean(librosa.feature.spectral_contrast(S=stft, sr=sr).T, axis=0)
    tonnetz = np.mean(librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr).T, axis=0)
    
    features = np.hstack([mfcc, chroma, mel, contrast, tonnetz])
    
    # 4. Scale
    # Reshape to (1, 193) because model expects a batch
    features = features.reshape(1, -1)
    features_scaled = scaler.transform(features)
    
    return features_scaled

In [3]:
# Scoring Algorithm
def calculate_confidence(emotion_probabilities, class_labels):
    """
    Calculates a weighted confidence score (0-100) based on emotion probabilities.
    """
    # Define your "Map" weights
    emotion_weights = {
        'happy': 1.0,
        'surprised': 0.9,
        'neutral': 0.8,
        'angry': 0.6,      # Confident but aggressive
        'disgust': 0.4,
        'sad': 0.3,
        'fearful': 0.2
    }
    
    score = 0
    
    # Dot product: Probability * Weight
    for i, emotion in enumerate(class_labels):
        prob = emotion_probabilities[i]
        weight = emotion_weights.get(emotion, 0.5)
        score += prob * weight
        
    return round(score * 100, 2)

# Test the function conceptually
test_probs = [0.1, 0.0, 0.0, 0.8, 0.1, 0.0, 0.0] # Mostly Happy
print(f"Simulated Score (Happy): {calculate_confidence(test_probs, classes)}/100")

Simulated Score (Happy): 94.0/100


In [4]:
# Sample Run
import pandas as pd
import random

# Pick a random file from our metadata
df = pd.read_csv("../data/processed/metadata.csv")
random_row = df.sample(1).iloc[0]
test_path = random_row['path']
true_emotion = random_row['emotion']

print(f"Testing File: {test_path}")
print(f"True Emotion: {true_emotion}")

# 1. Process
input_features = process_audio(test_path)

# 2. Predict
preds = model.predict(input_features)
pred_class_index = np.argmax(preds)
pred_emotion = classes[pred_class_index]
confidence_score = calculate_confidence(preds[0], classes)

# 3. Report
print(f"\nPredicted Emotion: {pred_emotion}")
print(f"Confidence Score: {confidence_score}/100")
print("-" * 30)
print("Probabilities:")
for i, emotion in enumerate(classes):
    print(f"  {emotion}: {preds[0][i]:.4f}")

Testing File: ../data/raw\TESS\YAF_fear\YAF_thumb_fear.wav
True Emotion: fearful
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 321ms/step

Predicted Emotion: fearful
Confidence Score: 20.030000686645508/100
------------------------------
Probabilities:
  angry: 0.0000
  disgust: 0.0001
  fearful: 0.9993
  happy: 0.0002
  neutral: 0.0000
  sad: 0.0003
  surprised: 0.0000


In [5]:
# Sample Run 2
import pandas as pd
import random

# Pick a random file from our metadata
df = pd.read_csv("../data/processed/metadata.csv")
random_row = df.sample(1).iloc[0]
test_path = random_row['path']
true_emotion = random_row['emotion']

print(f"Testing File: {test_path}")
print(f"True Emotion: {true_emotion}")

# 1. Process
input_features = process_audio(test_path)

# 2. Predict
preds = model.predict(input_features)
pred_class_index = np.argmax(preds)
pred_emotion = classes[pred_class_index]
confidence_score = calculate_confidence(preds[0], classes)

# 3. Report
print(f"\nPredicted Emotion: {pred_emotion}")
print(f"Confidence Score: {confidence_score}/100")
print("-" * 30)
print("Probabilities:")
for i, emotion in enumerate(classes):
    print(f"  {emotion}: {preds[0][i]:.4f}")

Testing File: ../data/raw\TESS\YAF_disgust\YAF_jail_disgust.wav
True Emotion: disgust
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step

Predicted Emotion: disgust
Confidence Score: 40.04999923706055/100
------------------------------
Probabilities:
  angry: 0.0005
  disgust: 0.9982
  fearful: 0.0001
  happy: 0.0006
  neutral: 0.0000
  sad: 0.0004
  surprised: 0.0001


In [6]:
# Sample Run 3
import pandas as pd
import random

# Pick a random file from our metadata
df = pd.read_csv("../data/processed/metadata.csv")
random_row = df.sample(1).iloc[0]
test_path = random_row['path']
true_emotion = random_row['emotion']

print(f"Testing File: {test_path}")
print(f"True Emotion: {true_emotion}")

# 1. Process
input_features = process_audio(test_path)

# 2. Predict
preds = model.predict(input_features)
pred_class_index = np.argmax(preds)
pred_emotion = classes[pred_class_index]
confidence_score = calculate_confidence(preds[0], classes)

# 3. Report
print(f"\nPredicted Emotion: {pred_emotion}")
print(f"Confidence Score: {confidence_score}/100")
print("-" * 30)
print("Probabilities:")
for i, emotion in enumerate(classes):
    print(f"  {emotion}: {preds[0][i]:.4f}")

Testing File: ../data/raw\RAVDESS\Actor_17\03-01-03-01-02-02-17.wav
True Emotion: happy
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step

Predicted Emotion: disgust
Confidence Score: 45.5099983215332/100
------------------------------
Probabilities:
  angry: 0.0705
  disgust: 0.3715
  fearful: 0.2250
  happy: 0.1580
  neutral: 0.0146
  sad: 0.1581
  surprised: 0.0023


In [7]:
# Sample Run 4
import pandas as pd
import random

# Pick a random file from our metadata
df = pd.read_csv("../data/processed/metadata.csv")
random_row = df.sample(1).iloc[0]
test_path = random_row['path']
true_emotion = random_row['emotion']

print(f"Testing File: {test_path}")
print(f"True Emotion: {true_emotion}")

# 1. Process
input_features = process_audio(test_path)

# 2. Predict
preds = model.predict(input_features)
pred_class_index = np.argmax(preds)
pred_emotion = classes[pred_class_index]
confidence_score = calculate_confidence(preds[0], classes)

# 3. Report
print(f"\nPredicted Emotion: {pred_emotion}")
print(f"Confidence Score: {confidence_score}/100")
print("-" * 30)
print("Probabilities:")
for i, emotion in enumerate(classes):
    print(f"  {emotion}: {preds[0][i]:.4f}")

Testing File: ../data/raw\RAVDESS\Actor_18\03-01-04-02-01-01-18.wav
True Emotion: sad
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step

Predicted Emotion: sad
Confidence Score: 31.139999389648438/100
------------------------------
Probabilities:
  angry: 0.0015
  disgust: 0.0219
  fearful: 0.0133
  happy: 0.0133
  neutral: 0.0012
  sad: 0.9485
  surprised: 0.0004


In [8]:
# Sample Run 5
import pandas as pd
import random

# Pick a random file from our metadata
df = pd.read_csv("../data/processed/metadata.csv")
random_row = df.sample(1).iloc[0]
test_path = random_row['path']
true_emotion = random_row['emotion']

print(f"Testing File: {test_path}")
print(f"True Emotion: {true_emotion}")

# 1. Process
input_features = process_audio(test_path)

# 2. Predict
preds = model.predict(input_features)
pred_class_index = np.argmax(preds)
pred_emotion = classes[pred_class_index]
confidence_score = calculate_confidence(preds[0], classes)

# 3. Report
print(f"\nPredicted Emotion: {pred_emotion}")
print(f"Confidence Score: {confidence_score}/100")
print("-" * 30)
print("Probabilities:")
for i, emotion in enumerate(classes):
    print(f"  {emotion}: {preds[0][i]:.4f}")

Testing File: ../data/raw\TESS\OAF_Pleasant_surprise\OAF_cab_ps.wav
True Emotion: surprised
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step

Predicted Emotion: surprised
Confidence Score: 89.98999786376953/100
------------------------------
Probabilities:
  angry: 0.0000
  disgust: 0.0001
  fearful: 0.0000
  happy: 0.0002
  neutral: 0.0000
  sad: 0.0000
  surprised: 0.9996


In [9]:
# Sample Run 6
import pandas as pd
import random

# Pick a random file from our metadata
df = pd.read_csv("../data/processed/metadata.csv")
random_row = df.sample(1).iloc[0]
test_path = random_row['path']
true_emotion = random_row['emotion']

print(f"Testing File: {test_path}")
print(f"True Emotion: {true_emotion}")

# 1. Process
input_features = process_audio(test_path)

# 2. Predict
preds = model.predict(input_features)
pred_class_index = np.argmax(preds)
pred_emotion = classes[pred_class_index]
confidence_score = calculate_confidence(preds[0], classes)

# 3. Report
print(f"\nPredicted Emotion: {pred_emotion}")
print(f"Confidence Score: {confidence_score}/100")
print("-" * 30)
print("Probabilities:")
for i, emotion in enumerate(classes):
    print(f"  {emotion}: {preds[0][i]:.4f}")

Testing File: ../data/raw\CREMA-D\1072_IOM_HAP_XX.wav
True Emotion: happy
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step

Predicted Emotion: happy
Confidence Score: 93.63999938964844/100
------------------------------
Probabilities:
  angry: 0.0093
  disgust: 0.0008
  fearful: 0.0728
  happy: 0.9152
  neutral: 0.0003
  sad: 0.0015
  surprised: 0.0001


In [10]:
# Sample Run 8
import pandas as pd
import random

# Pick a random file from our metadata
df = pd.read_csv("../data/processed/metadata.csv")
random_row = df.sample(1).iloc[0]
test_path = random_row['path']
true_emotion = random_row['emotion']

print(f"Testing File: {test_path}")
print(f"True Emotion: {true_emotion}")

# 1. Process
input_features = process_audio(test_path)

# 2. Predict
preds = model.predict(input_features)
pred_class_index = np.argmax(preds)
pred_emotion = classes[pred_class_index]
confidence_score = calculate_confidence(preds[0], classes)

# 3. Report
print(f"\nPredicted Emotion: {pred_emotion}")
print(f"Confidence Score: {confidence_score}/100")
print("-" * 30)
print("Probabilities:")
for i, emotion in enumerate(classes):
    print(f"  {emotion}: {preds[0][i]:.4f}")

Testing File: ../data/raw\CREMA-D\1073_IOM_FEA_XX.wav
True Emotion: fearful
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step

Predicted Emotion: disgust
Confidence Score: 44.41999816894531/100
------------------------------
Probabilities:
  angry: 0.0136
  disgust: 0.2931
  fearful: 0.2607
  happy: 0.0718
  neutral: 0.1731
  sad: 0.1876
  surprised: 0.0002


In [11]:
# Sample Run 9
import pandas as pd
import random

# Pick a random file from our metadata
df = pd.read_csv("../data/processed/metadata.csv")
random_row = df.sample(1).iloc[0]
test_path = random_row['path']
true_emotion = random_row['emotion']

print(f"Testing File: {test_path}")
print(f"True Emotion: {true_emotion}")

# 1. Process
input_features = process_audio(test_path)

# 2. Predict
preds = model.predict(input_features)
pred_class_index = np.argmax(preds)
pred_emotion = classes[pred_class_index]
confidence_score = calculate_confidence(preds[0], classes)

# 3. Report
print(f"\nPredicted Emotion: {pred_emotion}")
print(f"Confidence Score: {confidence_score}/100")
print("-" * 30)
print("Probabilities:")
for i, emotion in enumerate(classes):
    print(f"  {emotion}: {preds[0][i]:.4f}")

Testing File: ../data/raw\CREMA-D\1030_TAI_SAD_XX.wav
True Emotion: sad
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step

Predicted Emotion: disgust
Confidence Score: 38.68000030517578/100
------------------------------
Probabilities:
  angry: 0.0187
  disgust: 0.6786
  fearful: 0.0766
  happy: 0.0094
  neutral: 0.0286
  sad: 0.1880
  surprised: 0.0001


In [12]:
# Sample Run 10
import pandas as pd
import random

# Pick a random file from our metadata
df = pd.read_csv("../data/processed/metadata.csv")
random_row = df.sample(1).iloc[0]
test_path = random_row['path']
true_emotion = random_row['emotion']

print(f"Testing File: {test_path}")
print(f"True Emotion: {true_emotion}")

# 1. Process
input_features = process_audio(test_path)

# 2. Predict
preds = model.predict(input_features)
pred_class_index = np.argmax(preds)
pred_emotion = classes[pred_class_index]
confidence_score = calculate_confidence(preds[0], classes)

# 3. Report
print(f"\nPredicted Emotion: {pred_emotion}")
print(f"Confidence Score: {confidence_score}/100")
print("-" * 30)
print("Probabilities:")
for i, emotion in enumerate(classes):
    print(f"  {emotion}: {preds[0][i]:.4f}")

Testing File: ../data/raw\CREMA-D\1067_DFA_DIS_XX.wav
True Emotion: disgust
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step

Predicted Emotion: disgust
Confidence Score: 57.400001525878906/100
------------------------------
Probabilities:
  angry: 0.0193
  disgust: 0.3016
  fearful: 0.0938
  happy: 0.1556
  neutral: 0.2765
  sad: 0.1527
  surprised: 0.0004


In [16]:
# Enagement Function
def calculate_engagement(file_path):
    try:
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        
        # Energy Variance (Volume changes)
        rms = librosa.feature.rms(y=y)[0]
        energy_std = np.std(rms)
        
        # Pitch/Tone Variance (Vocal inflection proxy)
        centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        pitch_std = np.std(centroids)
        
        # Heuristic formula for prototype (scales raw variance to a 0-100 score)
        raw_engagement = (energy_std * 1000) + (pitch_std / 20)
        engagement_score = min(max(raw_engagement, 0), 100)
        
        return round(engagement_score, 2)
        
    except Exception as e:
        print(f"Error calculating metrics: {e}")
        return 0.0

In [15]:
# Report Function
def generate_full_acoustic_profile(file_path):
    input_features = process_audio(file_path)
    preds = model.predict(input_features, verbose=0)
    
    pred_class_index = np.argmax(preds)
    predicted_emotion = classes[pred_class_index]
    confidence_score = calculate_confidence(preds[0], classes)
    engagement_score = calculate_engagement(file_path)
    
    profile = {
        "detected_tone": predicted_emotion,
        "confidence_score": confidence_score,
        "engagement_score": engagement_score
    }
    
    return profile

In [18]:
# Sample Run
import random

# Randomly select 10 rows from the dataset
sample_rows = df.sample(10)

for index, row in sample_rows.iterrows():
    test_file = row['path']
    true_emotion = row['emotion']
    
    print(f"File Path: {test_file}")
    print(f"True Emotion (from dataset): {true_emotion.upper()}")
    
    try:
        # Run the full profile function
        profile = generate_full_acoustic_profile(test_file)
        
        print("IntWiz Profile Output:")
        for key, value in profile.items():
            # Formatting the keys to look nice (e.g., 'confidence_score' -> 'Confidence Score')
            print(f"  - {key.replace('_', ' ').title()}: {value}")
            
    except Exception as e:
        print(f"  - Error processing file: {e}")
        
    print("-" * 40)

File Path: ../data/raw\TESS\YAF_angry\YAF_gas_angry.wav
True Emotion (from dataset): ANGRY
IntWiz Profile Output:
  - Detected Tone: angry
  - Confidence Score: 59.939998626708984
  - Engagement Score: 100
----------------------------------------
File Path: ../data/raw\CREMA-D\1076_IWL_DIS_XX.wav
True Emotion (from dataset): DISGUST
IntWiz Profile Output:
  - Detected Tone: sad
  - Confidence Score: 49.029998779296875
  - Engagement Score: 19.91
----------------------------------------
File Path: ../data/raw\TESS\YAF_disgust\YAF_laud_disgust.wav
True Emotion (from dataset): DISGUST
IntWiz Profile Output:
  - Detected Tone: disgust
  - Confidence Score: 40.04999923706055
  - Engagement Score: 100
----------------------------------------
File Path: ../data/raw\CREMA-D\1037_ITS_HAP_XX.wav
True Emotion (from dataset): HAPPY
IntWiz Profile Output:
  - Detected Tone: fearful
  - Confidence Score: 54.5099983215332
  - Engagement Score: 50.66
----------------------------------------
File Path: